# Feature Extraction using Transfer Learning (ResNet-50)

This notebook implements feature extraction from preprocessed DeepFashion2 images using a pre-trained ResNet-50 model. The extracted features will be saved as embeddings for downstream tasks like similarity search and classification.

## Import Libraries

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.models as models

from tqdm import tqdm

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.83 GB


## Set Device and Paths

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Paths for Kaggle environment
ROOT_DIR = "/kaggle/input/preprocessed-deepfashion/output/"
INPUT_DIR = os.path.join(ROOT_DIR, "input/")
RESIZED_DIR = os.path.join(ROOT_DIR, "resized/")  # Path to resized images

# Create features directory in Kaggle working directory (writable location)
os.makedirs('/kaggle/working/FEATURES_DIR/', exist_ok=True)
FEATURES_DIR = '/kaggle/working/FEATURES_DIR/'

# CSV paths
TRAIN_CSV = os.path.join(INPUT_DIR, "train.csv")
VALIDATION_CSV = os.path.join(INPUT_DIR, "validation.csv")
TEST_CSV = os.path.join(INPUT_DIR, "test.csv")

print(f"\nData paths configured:")
print(f"  Input CSV directory: {INPUT_DIR}")
print(f"  Resized images directory: {RESIZED_DIR}")
print(f"  Features output directory: {FEATURES_DIR}")

Using device: cuda

Data paths configured:
  Input CSV directory: /kaggle/input/preprocessed-deepfashion/output/input/
  Resized images directory: /kaggle/input/preprocessed-deepfashion/output/resized/
  Features output directory: /kaggle/working/FEATURES_DIR/


## Load Preprocessed Data

In [3]:
# Load preprocessed dataframes
train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Loaded datasets:")
print(f"  Training samples: {len(train_df)}")
print(f"  Validation samples: {len(validation_df)}")
print(f"  Test samples: {len(test_df)}")

print(f"\nTrain DataFrame columns: {train_df.columns.tolist()}")
print(f"\nSample row from train_df:")
display(train_df.head(2))

print(f"\nSample image path from CSV: {train_df['path'].iloc[0]}")

Loaded datasets:
  Training samples: 312186
  Validation samples: 52490
  Test samples: 62629

Train DataFrame columns: ['path', 'segmentation', 'landmarks', 'b_box', 'category_id', 'category_name', 'scale', 'viewpoint', 'occlusion', 'zoom_in', 'img_height', 'img_width']

Sample row from train_df:


,path,segmentation,landmarks,b_box,category_id,category_name,scale,viewpoint,occlusion,zoom_in,img_height,img_width
0,/home/vishn/Documents/5/ML/Code/Project/output...,"[[125.99999999999999, 60.199999999999996, 127....","[387, 176, 1, 371, 163, 1, 364, 166, 1, 360, 1...","[79.44999999999999, 54.25, 154.7, 133.35]",4,long sleeve outwear,2,2,2,1,224,224
1,/home/vishn/Documents/5/ML/Code/Project/output...,"[[133.35, 107.1, 116.89999999999999, 101.14999...","[297, 266, 1, 334, 289, 1, 381, 306, 1, 261, 3...","[73.14999999999999, 92.05, 135.45, 188.6499999...",8,trousers,2,2,2,1,224,224



Sample image path from CSV: /home/vishn/Documents/5/ML/Code/Project/output/resized/train/039350.jpg


## Update Image Paths for Kaggle Environment

The CSV files contain local paths. We need to update them to point to Kaggle's uploaded resized images.

In [4]:
def update_image_paths(df, resized_dir):
    """
    Update image paths to point to Kaggle's resized directory.
    
    Original path: /home/vishn/Documents/5/ML/Code/Project/output/resized/train/000001.jpg
    New path: /kaggle/input/preprocessed-deepfashion/output/resized/train/000001.jpg
    """
    def fix_path(old_path):
        # Extract the relative part after 'resized/'
        if 'resized' in old_path:
            parts = old_path.split('resized/')
            if len(parts) == 2:
                relative_path = parts[1]  # e.g., 'train/000001.jpg'
                new_path = os.path.join(resized_dir, relative_path)
                return new_path
        return old_path  # Return original if pattern doesn't match
    
    df['path'] = df['path'].apply(fix_path)
    return df

# Update paths for all dataframes
print("Updating image paths to Kaggle environment...")
train_df = update_image_paths(train_df, RESIZED_DIR)
validation_df = update_image_paths(validation_df, RESIZED_DIR)
test_df = update_image_paths(test_df, RESIZED_DIR)

print(f"\nUpdated sample path: {train_df['path'].iloc[0]}")

# Verify first image exists
sample_path = train_df['path'].iloc[0]
if os.path.exists(sample_path):
    print(f" Image path verified: File exists!")
else:
    print(f" Warning: Image not found at {sample_path}")
    print(f"  Please check your Kaggle dataset upload structure.")

Updating image paths to Kaggle environment...

Updated sample path: /kaggle/input/preprocessed-deepfashion/output/resized/train/039350.jpg
 Image path verified: File exists!


## Define ResNet-50 Feature Extractor

ResNet-50 is a 50-layer residual network pre-trained on ImageNet. We'll use it to extract 2048-dimensional feature vectors from fashion images. These features capture high-level visual information (color, texture, shape) that can be used for classification, similarity search, and other downstream tasks.

In [5]:
class ResNet50FeatureExtractor(nn.Module):
    """
    ResNet-50 Feature Extractor for Transfer Learning
    
    This model uses a pre-trained ResNet-50 (trained on ImageNet) to extract
    2048-dimensional feature vectors from images. The final classification
    layer is removed, and we extract features from the global average pooling layer.
    
    Args:
        pretrained (bool): Whether to load ImageNet pre-trained weights
        fine_tune (bool): Whether to allow gradient updates (True) or freeze weights (False)
        freeze_until_layer (str): Optionally freeze layers up to a specific layer name
                                   Options: 'layer1', 'layer2', 'layer3', 'layer4', None
    """
    
    def __init__(self, pretrained=True, fine_tune=True, freeze_until_layer=None):
        super(ResNet50FeatureExtractor, self).__init__()
        
        # Load pre-trained ResNet-50
        resnet = models.resnet50(pretrained=pretrained)
        
        # Remove the final fully connected layer (fc)
        # Keep everything up to and including the adaptive average pooling layer
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Set gradient requirements
        if not fine_tune:
            # Freeze all layers
            for param in self.backbone.parameters():
                param.requires_grad = False
        elif freeze_until_layer is not None:
            # Freeze layers selectively
            self._freeze_layers_until(resnet, freeze_until_layer)
        
        # Store layer names for reference
        self.layer_names = [name for name, _ in resnet.named_children() if name != 'fc']
        
        print(f"ResNet-50 Feature Extractor initialized:")
        print(f"  - Pre-trained weights: {pretrained}")
        print(f"  - Fine-tuning enabled: {fine_tune}")
        print(f"  - Frozen until layer: {freeze_until_layer}")
        print(f"  - Output feature dimension: 2048")
    
    def _freeze_layers_until(self, resnet, layer_name):
        """
        Freeze all layers up to and including the specified layer.
        
        Args:
            resnet: The original ResNet model
            layer_name: Name of the layer to freeze until ('layer1', 'layer2', 'layer3', 'layer4')
        """
        freeze = True
        layer_order = ['conv1', 'bn1', 'relu', 'maxpool', 'layer1', 'layer2', 'layer3', 'layer4']
        
        for name, child in resnet.named_children():
            if name == 'fc':  # Skip the final fc layer
                continue
            
            if freeze:
                for param in child.parameters():
                    param.requires_grad = False
                print(f"  Frozen layer: {name}")
            else:
                for param in child.parameters():
                    param.requires_grad = True
                print(f"  Trainable layer: {name}")
            
            # Stop freezing after reaching the specified layer
            if name == layer_name:
                freeze = False
    
    def forward(self, x):
        """
        Extract features from input images.
        
        Args:
            x: Input tensor of shape (batch_size, 3, H, W)
        
        Returns:
            features: Feature tensor of shape (batch_size, 2048)
        """
        features = self.backbone(x)  # Shape: (batch_size, 2048, 1, 1)
        features = features.view(features.size(0), -1)  # Flatten to (batch_size, 2048)
        return features
    
    def get_num_params(self):
        """Return the total number of parameters in the model."""
        return sum(p.numel() for p in self.parameters())
    
    def get_num_trainable_params(self):
        """Return the number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## Initialize Feature Extractor Model

In [6]:
# Initialize the feature extractor
# For feature extraction only (no training), set fine_tune=False to freeze all weights
feature_extractor = ResNet50FeatureExtractor(
    pretrained=True,
    fine_tune=False,  # Freeze all layers for pure feature extraction
    freeze_until_layer=None
)

# Move model to device
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()  # Set to evaluation mode

print(f"\nModel Statistics:")
print(f"  Total parameters: {feature_extractor.get_num_params():,}")
print(f"  Trainable parameters: {feature_extractor.get_num_trainable_params():,}")

# Test with a dummy input
dummy_input = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    dummy_output = feature_extractor(dummy_input)
print(f"\nTest output shape: {dummy_output.shape}")  # Should be (4, 2048)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


ResNet-50 Feature Extractor initialized:
  - Pre-trained weights: True
  - Fine-tuning enabled: False
  - Frozen until layer: None
  - Output feature dimension: 2048

Model Statistics:
  Total parameters: 23,508,032
  Trainable parameters: 0

Test output shape: torch.Size([4, 2048])


## Define Data Preprocessing and Dataset

In [7]:
# ImageNet normalization constants 
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Target size 
TARGET_SIZE = (224, 224)

# Training transforms WITH AUGMENTATION
train_transform_with_augmentation = transforms.Compose([
    transforms.ToPILImage(),  # Convert numpy/tensor to PIL
    transforms.RandomResizedCrop(
        TARGET_SIZE, 
        scale=(0.8, 1.0),  # Crop 80-100% of original
        ratio=(0.9, 1.1)   # Aspect ratio variation
    ),
    transforms.RandomHorizontalFlip(p=0.5),  # 50% chance of horizontal flip
    transforms.RandomRotation(degrees=15),    # Rotate ±15 degrees
    transforms.ColorJitter(
        brightness=0.2,  # Brightness variation ±20%
        contrast=0.2,    # Contrast variation ±20%
        saturation=0.2,  # Saturation variation ±20%
        hue=0.1          # Hue variation ±10%
    ),
    transforms.ToTensor(),  # Convert to tensor [0, 1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Validation/Test transforms WITHOUT AUGMENTATION 
val_transform_no_augmentation = transforms.Compose([
    transforms.ToPILImage(),  # Convert numpy array to PIL Image
    transforms.Resize(TARGET_SIZE),  # Resize to (224, 224)
    transforms.ToTensor(),  # Convert to tensor [0, 1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # Normalize with ImageNet stats
])



## Define Transform Pipelines (Matching preprocessing.ipynb)

We'll define both training transforms (with augmentation) and validation transforms (without augmentation).

In [8]:
class FashionFeatureDataset(Dataset):
    """
    Dataset for feature extraction from preprocessed DeepFashion2 images.
    """
    
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        """
        Args:
            dataframe: Pandas DataFrame with image paths and metadata
            transform: torchvision transforms to apply
        """
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = row['path']
        
        # Handle different possible image loading scenarios
        try:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Failed to load image: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a black image as fallback
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get image identifier
        img_name = os.path.basename(img_path)
        
        # Prepare metadata
        sample = {
            'image': image,
            'image_id': img_name,
            'image_path': img_path,
            'index': idx
        }
        
        # Add category info if available
        if 'category_id' in row:
            sample['category_id'] = row['category_id']
        if 'category_name' in row:
            sample['category_name'] = row['category_name']
        if 'style' in row:
            sample['style'] = row['style']
        
        return sample


## Create Datasets and DataLoaders

In [9]:
# Create datasets
# Use validation transforms (NO augmentation) 
# train_dataset = FashionFeatureDataset(train_df, transform=val_transform_no_augmentation)
# val_dataset = FashionFeatureDataset(validation_df, transform=val_transform_no_augmentation)
# test_dataset = FashionFeatureDataset(test_df, transform=val_transform_no_augmentation)



#use augmentation on training set
train_dataset = FashionFeatureDataset(train_df, transform=train_transform_with_augmentation)
val_dataset = FashionFeatureDataset(validation_df, transform=val_transform_no_augmentation)
test_dataset = FashionFeatureDataset(test_df, transform=val_transform_no_augmentation)

print(f"\n  Train: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Batch size for feature extraction
BATCH_SIZE = 64

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Keep order for easier matching with dataframe
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"\nDataLoaders created with batch_size={BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")


  Train: 312186 samples
  Validation: 52490 samples
  Test: 62629 samples

DataLoaders created with batch_size=64
  Train batches: 4878
  Validation batches: 821
  Test batches: 979


## Feature Extraction Function

In [10]:
def extract_features(model, dataloader, device, desc="Extracting features"):
    """
    Extract features from all images in the dataloader.
    
    Args:
        model: Feature extraction model (ResNet50FeatureExtractor)
        dataloader: PyTorch DataLoader
        device: torch.device (cuda or cpu)
        desc: Description for the progress bar
    
    Returns:
        features_array: Numpy array of shape (num_samples, 2048)
        metadata_list: List of dictionaries containing metadata for each sample
    """
    model.eval()
    
    all_features = []
    all_metadata = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            # Move images to device
            images = batch['image'].to(device)
            
            # Extract features
            features = model(images)  # Shape: (batch_size, 2048)
            
            # Move to CPU and convert to numpy
            features_np = features.cpu().numpy()
            all_features.append(features_np)
            
            # Store metadata
            batch_size = len(batch['image_id'])
            for i in range(batch_size):
                metadata = {
                    'image_id': batch['image_id'][i],
                    'image_path': batch['image_path'][i],
                    'index': batch['index'][i].item()
                }
                
                # Add category info if available
                if 'category_id' in batch:
                    metadata['category_id'] = batch['category_id'][i].item()
                if 'category_name' in batch:
                    metadata['category_name'] = batch['category_name'][i]
                if 'style' in batch:
                    metadata['style'] = batch['style'][i].item()
                
                all_metadata.append(metadata)
    
    # Concatenate all features
    features_array = np.vstack(all_features)
    
    print(f"\nFeature extraction complete!")
    print(f"  Features shape: {features_array.shape}")
    print(f"  Metadata entries: {len(all_metadata)}")
    
    return features_array, all_metadata


## Extract Features for All Datasets

In [11]:
# Extract features for training set
print("=" * 80)
print("EXTRACTING FEATURES FOR TRAINING SET")
print("=" * 80)
train_features, train_metadata = extract_features(
    feature_extractor, 
    train_loader, 
    device, 
    desc="Train set"
)

EXTRACTING FEATURES FOR TRAINING SET


Train set: 100%|██████████| 4878/4878 [37:53<00:00,  2.15it/s]



Feature extraction complete!
  Features shape: (312186, 2048)
  Metadata entries: 312186


In [12]:
# Extract features for validation set
print("\n" + "=" * 80)
print("EXTRACTING FEATURES FOR VALIDATION SET")
print("=" * 80)
val_features, val_metadata = extract_features(
    feature_extractor, 
    val_loader, 
    device, 
    desc="Validation set"
)


EXTRACTING FEATURES FOR VALIDATION SET


Validation set: 100%|██████████| 821/821 [03:17<00:00,  4.16it/s]



Feature extraction complete!
  Features shape: (52490, 2048)
  Metadata entries: 52490


In [13]:
# Extract features for test set
print("\n" + "=" * 80)
print("EXTRACTING FEATURES FOR TEST SET")
print("=" * 80)
test_features, test_metadata = extract_features(
    feature_extractor, 
    test_loader, 
    device, 
    desc="Test set"
)


EXTRACTING FEATURES FOR TEST SET


Test set: 100%|██████████| 979/979 [04:39<00:00,  3.50it/s]



Feature extraction complete!
  Features shape: (62629, 2048)
  Metadata entries: 62629


## Save Features to CSV Files

We'll create new CSV files that include the original data plus the extracted 2048-dimensional features.

In [14]:
def save_features_to_csv(original_df, features_array, output_path, dataset_name):
    """
    Save extracted features along with original dataframe information to CSV.
    
    Args:
        original_df: Original pandas DataFrame
        features_array: Numpy array of extracted features (num_samples, 2048)
        output_path: Path to save the output CSV
        dataset_name: Name of the dataset (for logging)
    
    Returns:
        df_with_features: DataFrame with original columns + feature columns
    """
    print(f"\nSaving {dataset_name} features to CSV...")
    
    # Create a copy of the original dataframe
    df_with_features = original_df.copy()
    
    # Add each feature dimension as a separate column
    for i in range(features_array.shape[1]):
        df_with_features[f'feature_{i}'] = features_array[:, i]
    
    # Save to CSV
    df_with_features.to_csv(output_path, index=False)
    
    print(f"  Saved to: {output_path}")
    print(f"  Shape: {df_with_features.shape}")
    print(f"  Columns: {len(df_with_features.columns)} ({len(original_df.columns)} original + {features_array.shape[1]} features)")
    
    # Display file size
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"  File size: {file_size_mb:.2f} MB")
    
    return df_with_features


In [15]:
# Save features for all datasets
print("=" * 80)
print("SAVING EXTRACTED FEATURES")
print("=" * 80)

train_features_csv = os.path.join(FEATURES_DIR, "train_with_features.csv")
val_features_csv = os.path.join(FEATURES_DIR, "validation_with_features.csv")
test_features_csv = os.path.join(FEATURES_DIR, "test_with_features.csv")

# Save training features
train_df_with_features = save_features_to_csv(
    train_df, 
    train_features,
    train_features_csv,
    "Training"
)

# Save validation features
val_df_with_features = save_features_to_csv(
    validation_df, 
    val_features,
    val_features_csv,
    "Validation"
)

# Save test features
test_df_with_features = save_features_to_csv(
    test_df, 
    test_features,
    test_features_csv,
    "Test"
)


SAVING EXTRACTED FEATURES

Saving Training features to CSV...
  Saved to: /kaggle/working/FEATURES_DIR/train_with_features.csv
  Shape: (312186, 2060)
  Columns: 2060 (12 original + 2048 features)
  File size: 7024.59 MB

Saving Validation features to CSV...
  Saved to: /kaggle/working/FEATURES_DIR/validation_with_features.csv
  Shape: (52490, 2060)
  Columns: 2060 (12 original + 2048 features)
  File size: 1296.30 MB

Saving Test features to CSV...
  Saved to: /kaggle/working/FEATURES_DIR/test_with_features.csv
  Shape: (62629, 2051)
  Columns: 2051 (3 original + 2048 features)
  File size: 1310.29 MB


## Save Features as Numpy Arrays (for faster loading)

In [16]:
# Save as numpy arrays for faster loading in future
np.save(os.path.join(FEATURES_DIR, 'train_features.npy'), train_features)
np.save(os.path.join(FEATURES_DIR, 'val_features.npy'), val_features)
np.save(os.path.join(FEATURES_DIR, 'test_features.npy'), test_features)
